In [2]:
import os
if not os.path.exists('IRkernel_installed'):
  # Install R kernel and dependencies
  !R -e 'install.packages(c("repr", "IRdisplay", "evaluate", "crayon", "pbdZMQ", "devtools", "uuid", "digest", "remotes"))'
  !R -e 'remotes::install_github("IRkernel/IRkernel")'
  !R -e 'IRkernel::installspec(user = FALSE)'
  !touch IRkernel_installed


After running the cell above, you will need to restart the runtime and then change the notebook settings:

1. Go to `Runtime` -> `Restart runtime`.
2. Go to `Runtime` -> `Change runtime type`.
3. In the dropdown, select `R` and click `Save`.

Once done, you can start writing R code directly in your cells.

In [1]:
install.packages(c("naniar", "skimr"))

Installing packages into ‘/usr/local/lib/R/site-library’
(as ‘lib’ is unspecified)

also installing the dependencies ‘gridExtra’, ‘plyr’, ‘norm’, ‘visdat’, ‘viridis’, ‘UpSetR’




In [9]:
# This cell was for installing 'pdftools' when in Python mode.
# With the notebook now in R mode, packages should be installed directly using install.packages().
# This cell is now commented out as it's no longer necessary.

Now that you have the `pdftools` package installed, you can use it to read the content of your PDF file. Remember to switch to an R code cell to use R syntax directly.

Here's how to read the PDF content:

In [10]:
# This cell was for loading 'pdftools' and reading PDF content when in Python mode.
# With the notebook now in R mode, you can use R syntax directly.
# This cell is now commented out as its original purpose with `!R -e` is no longer applicable.

In [2]:
library(naniar)
library(skimr)


Attaching package: ‘skimr’


The following object is masked from ‘package:naniar’:

    n_complete




In [ ]:
# Assignment 2 – R Programming Lab

## Lab 3: Control Flow for Data Cleaning

### Aim
To clean healthcare data using R functions, if-else conditions,
error handling, loops, and vectorized operations.

In [3]:
# Install packages if required
install.packages("dplyr")

Installing package into ‘/usr/local/lib/R/site-library’
(as ‘lib’ is unspecified)



In [4]:
library(dplyr)


Attaching package: ‘dplyr’


The following objects are masked from ‘package:stats’:

    filter, lag


The following objects are masked from ‘package:base’:

    intersect, setdiff, setequal, union




In [12]:
# Install/read required package
install.packages("readr")
library(readr)

Installing package into ‘/usr/local/lib/R/site-library’
(as ‘lib’ is unspecified)



In [16]:
# Create a sample Heart Disease dataset

set.seed(123)

heart_data <- data.frame(
  age = sample(30:75, 1000, replace = TRUE),

  trestbps = sample(
    90:200,
    1000,
    replace = TRUE
  ),

  chol = sample(
    150:400,
    1000,
    replace = TRUE
  ),

  maxhr = sample(
    100:200,
    1000,
    replace = TRUE
  )
)

# Display first 6 rows
head(heart_data)

,age,trestbps,chol,maxhr
,<int>,<int>,<int>,<int>
1,60,124,224,109
2,44,119,279,136
3,43,123,277,117
4,32,158,337,177
5,71,158,296,121
6,72,160,389,192


In [17]:
# Check structure
str(heart_data)

# Check dimensions
dim(heart_data)

# Summary statistics
summary(heart_data)

'data.frame':	1000 obs. of  4 variables:
 $ age     : int  60 44 43 32 71 72 66 43 54 55 ...
 $ trestbps: int  124 119 123 158 158 160 91 169 193 112 ...
 $ chol    : int  224 279 277 337 296 389 343 270 190 364 ...
 $ maxhr   : int  109 136 117 177 121 192 118 182 149 137 ...


[1] 1000    4

      age          trestbps          chol           maxhr      
 Min.   :30.0   Min.   : 90.0   Min.   :150.0   Min.   :100.0  
 1st Qu.:41.0   1st Qu.:117.0   1st Qu.:209.0   1st Qu.:123.0  
 Median :53.0   Median :145.0   Median :268.5   Median :147.0  
 Mean   :52.6   Mean   :145.4   Mean   :272.0   Mean   :148.7  
 3rd Qu.:64.0   3rd Qu.:172.0   3rd Qu.:336.0   3rd Qu.:174.0  
 Max.   :75.0   Max.   :200.0   Max.   :400.0   Max.   :200.0  

In [19]:
# Introduce negative BP values
heart_data$trestbps[c(5, 10, 15)] <- c(-20, -50, -10)

# Introduce missing BP values
heart_data$trestbps[c(20, 25, 30)] <- NA

# Introduce extreme BP values
heart_data$trestbps[c(35, 40, 45)] <- c(320, 350, 400)

In [20]:
# Display the modified observations
heart_data[c(5, 10, 15, 20, 25, 30, 35, 40, 45), ]

,age,trestbps,chol,maxhr
,<int>,<dbl>,<int>,<int>
5,71,-20,296,121
10,55,-50,364,137
15,38,-10,358,111
20,36,NA,291,173
25,43,NA,351,153
30,44,NA,280,178
35,38,320,179,135
40,36,350,324,178
45,63,400,260,162


In [21]:
# BP Cleaning Function

clean_bp <- function(bp) {

  if (is.na(bp)) {
    return(NA)

  } else if (bp < 0) {
    return(NA)

  } else if (bp > 250) {
    return(250)

  } else {
    return(bp)
  }
}

In [22]:
# Test the BP cleaning function

clean_bp(-20)
clean_bp(320)
clean_bp(120)
clean_bp(NA)

[1] NA

[1] 250

[1] 120

[1] NA

In [23]:
# Apply cleaning function to all BP values

heart_data$trestbps_clean <- sapply(
  heart_data$trestbps,
  clean_bp
)

# Display original and cleaned BP
head(
  heart_data[, c("trestbps", "trestbps_clean")],
  20
)

,trestbps,trestbps_clean
,<dbl>,<dbl>
1,124,124
2,119,119
3,123,123
4,158,158
5,-20,NA
6,160,160
7,91,91
8,169,169
9,193,193


In [24]:
# Calculate mean BP safely using tryCatch()

mean_bp <- tryCatch({

  mean(heart_data$trestbps_clean, na.rm = TRUE)

}, warning = function(w) {

  message("Warning while calculating mean BP: ", w$message)
  NA

}, error = function(e) {

  message("Error while calculating mean BP: ", e$message)
  NA
})

mean_bp

[1] 145.8491

In [25]:
mean(heart_data$trestbps_clean)

[1] NA

In [26]:
mean(heart_data$trestbps_clean, na.rm = TRUE)

[1] 145.8491

In [ ]:
chol / trestbps

In [27]:
# Calculate cholesterol / BP ratio safely

calculate_ratio <- function(chol, bp) {

  tryCatch({

    if (is.na(chol) || is.na(bp)) {
      stop("Cholesterol or BP is missing.")
    }

    if (bp == 0) {
      stop("Cannot divide by zero.")
    }

    if (bp < 0) {
      stop("BP cannot be negative.")
    }

    ratio <- chol / bp

    return(ratio)

  }, error = function(e) {

    message("Invalid ratio: ", e$message)
    return(NA)
  })
}

In [28]:
# Valid values
calculate_ratio(200, 120)

[1] 1.666667

In [29]:
calculate_ratio(200, 0)

Invalid ratio: Cannot divide by zero.



[1] NA

In [30]:
calculate_ratio(200, NA)

Invalid ratio: Cholesterol or BP is missing.



[1] NA

In [31]:
calculate_ratio(200, -50)

Invalid ratio: BP cannot be negative.



[1] NA

In [32]:
# Calculate cholesterol / BP ratio for all patients

heart_data$chol_bp_ratio <- mapply(
  calculate_ratio,
  heart_data$chol,
  heart_data$trestbps_clean
)

# View results
head(
  heart_data[, c("chol", "trestbps_clean", "chol_bp_ratio")],
  10
)

Invalid ratio: Cholesterol or BP is missing.

Invalid ratio: Cholesterol or BP is missing.

Invalid ratio: Cholesterol or BP is missing.

Invalid ratio: Cholesterol or BP is missing.

Invalid ratio: Cholesterol or BP is missing.

Invalid ratio: Cholesterol or BP is missing.



,chol,trestbps_clean,chol_bp_ratio
,<int>,<dbl>,<dbl>
1,224,124,1.806452
2,279,119,2.344538
3,277,123,2.252033
4,337,158,2.132911
5,296,NA,NA
6,389,160,2.431250
7,343,91,3.769231
8,270,169,1.597633
9,190,193,0.984456


In [33]:
# Loop-based BP cleaning

loop_clean_bp <- function(bp) {

  result <- bp

  for (i in 1:length(bp)) {

    if (is.na(bp[i])) {
      result[i] <- NA

    } else if (bp[i] < 0) {
      result[i] <- NA

    } else if (bp[i] > 250) {
      result[i] <- 250

    } else {
      result[i] <- bp[i]
    }
  }

  return(result)
}

In [34]:
loop_clean_bp(heart_data$trestbps)[1:10]

[1] 124 119 123 158  NA 160  91 169 193  NA

In [35]:
# Vectorized BP cleaning

vectorized_clean_bp <- function(bp) {

  result <- bp

  result[bp < 0] <- NA
  result[bp > 250] <- 250

  return(result)
}

In [36]:
vectorized_clean_bp(heart_data$trestbps)[1:10]

[1] 124 119 123 158  NA 160  91 169 193  NA

In [37]:
# Compare loop and vectorized results

loop_result <- loop_clean_bp(heart_data$trestbps)

vectorized_result <- vectorized_clean_bp(heart_data$trestbps)

identical(loop_result, vectorized_result)

[1] TRUE

In [38]:
# Measure loop execution time

loop_time <- system.time({
  for (j in 1:100) {
    loop_result <- loop_clean_bp(heart_data$trestbps)
  }
})

loop_time

   user  system elapsed 
  0.035   0.000   0.034 

In [39]:
# Measure vectorized execution time

vectorized_time <- system.time({
  for (j in 1:100) {
    vectorized_result <- vectorized_clean_bp(heart_data$trestbps)
  }
})

vectorized_time

   user  system elapsed 
  0.005   0.000   0.005 

In [40]:
# Compare execution times

comparison <- data.frame(
  Method = c("For Loop", "Vectorized"),
  Time = c(
    loop_time["user.self"],
    vectorized_time["user.self"]
  )
)

comparison

Method,Time
<chr>,<dbl>
For Loop,0.035
Vectorized,0.005


In [41]:
# Count missing BP values

missing_bp <- sum(is.na(heart_data$trestbps_clean))

missing_bp

[1] 6

In [42]:
# Minimum BP

min_bp <- min(
  heart_data$trestbps_clean,
  na.rm = TRUE
)

# Maximum BP

max_bp <- max(
  heart_data$trestbps_clean,
  na.rm = TRUE
)

# Mean BP

mean_bp <- mean(
  heart_data$trestbps_clean,
  na.rm = TRUE
)

# Median BP

median_bp <- median(
  heart_data$trestbps_clean,
  na.rm = TRUE
)

min_bp
max_bp
mean_bp
median_bp

[1] 90

[1] 250

[1] 145.8491

[1] 146

In [43]:
# Check for negative BP values

negative_values <- sum(
  heart_data$trestbps_clean < 0,
  na.rm = TRUE
)

# Check for BP values greater than 250

above_250 <- sum(
  heart_data$trestbps_clean > 250,
  na.rm = TRUE
)

negative_values
above_250

[1] 0

[1] 0

In [44]:
# Export cleaned dataset

write.csv(
  heart_data,
  "cleaned_heart_data.csv",
  row.names = FALSE
)

After executing the above code, the text content of your PDF assignment will be printed. You can then analyze this text to understand the problems and requirements for your R programming tasks. If you want to view the entire content, you can adjust `substr` or print `pdf_content` directly.